# Demo 1: Tracing

**Concept**: MLflow Tracing records every request as a **trace**, a tree of **spans**. Each span is one operation and captures its inputs, outputs, timing, type and status. Databricks' managed MLflow supports the standard MLflow SDK and is OpenTelemetry-compatible.

**We change very little**: same agent, plus four small additions (see `common/agent_traced.py`).

| Addition | What it gives you |
|---|---|
| `mlflow.openai.autolog()` | A `CHAT_MODEL` span for the LLM call, with no extra code |
| `@mlflow.trace(span_type="RETRIEVER")` | A span showing which wiki pages were retrieved |
| `mlflow.start_span(...)` | A manual span for our own logic (prompt building) |
| `@mlflow.trace(span_type="AGENT")` | A root span for the whole request |

In [0]:
%pip install -U "mlflow[databricks]>=3.1" openai databricks-sdk
dbutils.library.restartPython()

In [0]:
import os
import sys

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))

import mlflow

from common.config import EXPERIMENT_PATH

mlflow.set_tracking_uri("databricks")
# Traces are logged to an MLflow Experiment. It is the hub for everything in later demos.
# Set EXPERIMENT_PATH in common/config.py first. (If demo 6 will store traces in Unity Catalog, create the experiment with a UC trace location at creation time, not later.)
mlflow.set_experiment(EXPERIMENT_PATH)

If you are using MLflow Tracing, you can migrate your traces to Unity Catalog for unlimited storage, fine-grained access controls, and queryability from notebooks, SQL, and dashboards. Learn more: https://docs.databricks.com/aws/en/mlflow3/genai/tracing/migrate-traces-to-uc


<Experiment: artifact_location='dbfs:/databricks/mlflow-tracking/1571690280311094', creation_time=1792292632487, effective_trace_archival_retention=None, experiment_id='1571690280311094', last_update_time=1792292632487, lifecycle_stage='active', name='/Shared/python-wiki-agent-experiment', tags={'mlflow.experiment.sourceName': '/Shared/python-wiki-agent-experiment',
 'mlflow.experimentType': 'MLFLOW_EXPERIMENT',
 'mlflow.ownerEmail': 'pranav@platformatory.com',
 'mlflow.ownerId': '77162250982329'}, trace_location=None, workspace='default'>

## 1. Run the traced agent

In a Databricks notebook the trace is rendered right under the cell. Expand the tree.

In [0]:
from common.agent_traced import answer_question

result = answer_question("How do I add an item to a list?")
print(result["response"])

To add an item to a list, you can use the `append()` method. Here's an example:

```python
nums = [1, 2, 3]
nums.append(4)
print(nums)  # Output: [1, 2, 3, 4]
```

In this example, `nums.append(4)` adds the number 4 to the end of the list `nums`.


Trace(trace_id='tr-68315b30369ba1a888ea8abb608cba64')

## 2. What to look at in the trace

A typical trace looks like this (the root is the agent, the children are the steps):

```
answer_question          AGENT        <- whole request, total latency
├── search_wiki          RETRIEVER    <- which pages, with scores
├── build_prompt         (manual)     <- our own step
└── Completions          CHAT_MODEL   <- prompt, response, tokens
```

Click each span and check:
1. **Inputs / outputs**: what went in and came out
2. **Duration**: retrieval versus the LLM call
3. **Span type** and **status**

You can also open the experiment's **Traces** tab to see all requests in one list.

## 3. Break it

Now the question the wiki cannot answer. In demo 0 we could only see a fluent answer.

In [0]:
result = answer_question("How do I handle errors in Python?")
print(result["response"])

**Error Handling in Python**

Python provides a built-in mechanism for handling errors, also known as exceptions. There are two types of errors in Python: syntax errors and exceptions.

### Syntax Errors

Syntax errors occur when there is an error in the syntax of the code, such as a missing colon or a typo. These errors are usually caught by the interpreter before the code is executed.

### Exceptions

Exceptions are errors that occur during the execution of the code, such as dividing by zero or trying to open a file that does not exist. Python provides a `try`-`except` block to handle exceptions.

**Try-Except Block**
--------------------

The `try`-`except` block consists of two parts:

*   `try`: This block contains the code that might raise an exception.
*   `except`: This block contains the code that will be executed if an exception is raised.

**Example**
-----------

```python
try:
    # Code that might raise an exception
    x = 5 / 0
except ZeroDivisionError:
    # Code that 

Trace(trace_id='tr-408b75a885392275a03b1feb35cd746e')

Open the **`search_wiki`** span of this trace:

- It returned no pages (no overlap with the question).
- The prompt therefore contained no pages, and the model answered from its own memory.

The answer may be correct, but it is **not grounded in the wiki**, and now we can *see* that. We found the failure by hand-inspecting one trace. For ten questions that is fine; for ten thousand it is not.

**Next: demo 2 turns "is this answer good?" into an automated, repeatable measurement.**